In [ ]:
import pandas as pd
import torch
import json
import re
import os
from datasets import Dataset
from transformers import TrainingArguments
from tqdm import tqdm


In [ ]:

!pip install bitsandbytes transformers accelerate -q

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

torch.cuda.empty_cache()

model_name = "yandex/YandexGPT-5-Lite-8B-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("✅ Модель загружена!")

# Тест


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.5 MB/s eta 0:00:00


config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/192k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 2.57MB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

✅ Модель загружена!


In [ ]:
import torch

In [ ]:
torch.cuda.empty_cache()


In [ ]:
!pip install bitsandbytes -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 29.8 MB/s eta 0:00:00


In [ ]:
train_df = pd.read_excel("/content/cases_train2 (5).xlsx")
test_df = pd.read_excel("/content/Test1_2 (3).xlsx")


In [ ]:
def generate_critique(project_text, scores, model, tokenizer, max_new_tokens=1024):
    if pd.isna(project_text) or project_text == "":
        return "❌ Нет текста проекта"

    # Исправленный пример, соответствующий требуемой структуре
    example_critique = """
ПРИМЕР КАЧЕСТВЕННОЙ КРИТИКИ:

1. Анализ ЦА (оценка 1/5): Целевая аудитория не сегментирована и описана обобщенно как «таксопарки и водители». Отсутствует анализ конкретных сегментов (самозанятые водители, микропарки на 2-5 машин, крупные таксопарки), их проблем, барьеров и потребностей. Не представлена карта клиентского пути (CJM). Рекомендуется провести сегментацию ЦА и глубинные интервью с представителями каждого сегмента.

2. Проработка решения (оценка 1/5): Предложенное решение носит шаблонный характер («кредиты на авто и топливные карты») и не имеет уникальной концепции. Отсутствует MVP-стратегия, дорожная карта развития продукта. Партнеры упомянуты без конкретных условий сотрудничества. Необходимо разработать комплексное решение с уникальным УТП, включая кредит с отсрочкой, топливные карты и страховку.

3. Финансовая модель (оценка 1/5): Финансовые показатели (3 млн рублей, 100 клиентов) не обоснованы. Отсутствуют ключевые метрики: CAC (стоимость привлечения клиента), LTV (пожизненная ценность клиента), NPV (чистая приведенная стоимость), точка безубыточности. Нет сценарного анализа (оптимистичный, пессимистичный, реалистичный). Требуется детализированная финансовая модель с обоснованием всех показателей.

4. Анализ рисков (оценка 1/5): Риски полностью отсутствуют, что нереалистично для любого бизнес-проекта. Нет количественной оценки вероятности и влияния рисков, отсутствуют планы митигации. Рекомендуется описать основные риски (невозврат кредитов, аварийность, ценовая конкуренция) и разработать стратегии их снижения.

5. Доказательства (оценка 2/5): Единственным доказательством является знакомый таксист как источник информации, что не является валидным подтверждением. Упоминается статья без ссылки и конкретных данных. Необходимо провести исследование рынка с использованием данных Росстата, интервью с 10+ владельцами таксопарков, собрать реальные метрики и кейсы.

Итоговая оценка: 1.2/5
Общий вывод: Проект находится на начальной стадии концепции и требует фундаментальной проработки по всем критериям. Основные направления доработки: сегментация ЦА, создание уникального решения, построение детальной финансовой модели, анализ рисков и сбор доказательной базы через пилотные запуски и интервью.
"""

    system_prompt = f"""Ты — эксперт по оценке бизнес-проектов.

Вот оценки текущего проекта по 5 критериям (каждый от 1 до 5):
1. Анализ ЦА: {scores['ЦА']}/5
2. Проработка решения: {scores['Проработка']}/5
3. Финансовая модель: {scores['Финансы']}/5
4. Анализ рисков: {scores['Риски']}/5
5. Доказательства: {scores['Доказательства']}/5

{example_critique}

Напиши РАЗВЕРНУТУЮ КРИТИКУ для КОНКРЕТНОГО проекта, используя оценки выше.

КРИТИЧЕСКИ ВАЖНО:
- Пример выше показывает СТРУКТУРУ и СТИЛЬ, но НЕ КОПИРУЙ текст примера!
- Твоя критика должна быть УНИКАЛЬНОЙ для этого конкретного проекта
- Обязательно ссылайся на КОНКРЕТНЫЕ детали из текста проекта
- Объясни, ПОЧЕМУ проект получил каждую из указанных оценок
- Предложи КОНКРЕТНЫЕ улучшения, основанные на слабых местах проекта
- Формат должен ТОЧНО соответствовать структуре примера
- Не используй смайлики, слэнг, сохраняй нейтральный деловой стиль

Формат ответа (строго соблюдай структуру):
1. Анализ ЦА (оценка X/5): [уникальный комментарий по конкретному проекту]
2. Проработка решения (оценка X/5): [уникальный комментарий по конкретному проекту]
3. Финансовая модель (оценка X/5): [уникальный комментарий по конкретному проекту]
4. Анализ рисков (оценка X/5): [уникальный комментарий по конкретному проекту]
5. Доказательства (оценка X/5): [уникальный комментарий по конкретному проекту]

Итоговая оценка: {sum(scores.values()) / 5:.1f}/5
Общий вывод: [уникальный вывод по конкретному проекту]
"""

    prompt = f"""<|system|>
{system_prompt}
<|user|>
Вот текст проекта для оценки:

{project_text[:3000]}
<|assistant|>
"""

    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to("cuda")

    try:
        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids,
                max_new_tokens=max_new_tokens,
                temperature=0.8,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        if "<|assistant|>" in response:
            parts = response.split("<|assistant|>")
            if len(parts) > 1:
                return parts[-1].strip()

        return response

    except Exception as e:
        return f"❌ Ошибка: {str(e)}"

In [ ]:
test_df["Критика_модели"] = None


In [ ]:
for idx in tqdm(test_df.index, desc="Обработка кейсов"):
    project_text = test_df.loc[idx, "Решение кейса"]
    scores = {
        'ЦА': int(test_df.loc[idx, 'ЦА']) if 'ЦА' in test_df.columns else 3,
        'Проработка': int(test_df.loc[idx, 'Проработка решения']) if 'Проработка решения' in test_df.columns else 3,
        'Финансы': int(test_df.loc[idx, 'Финансовая модель и метрики']) if 'Финансовая модель и метрики' in test_df.columns else 3,
        'Риски': int(test_df.loc[idx, 'Анализ рисков']) if 'Анализ рисков' in test_df.columns else 3,
        'Доказательства': int(test_df.loc[idx, 'Доказательства']) if 'Доказательства' in test_df.columns else 3
    }
    critique = generate_critique(project_text, scores, model, tokenizer)
    test_df.loc[idx, "Критика_модели"] = critique
    if (idx + 1) % 5 == 0:
        test_df.to_excel("critique_partial.xlsx", index=False)
        print(f"\n💾 Сохранено {idx + 1} кейсов")


Обработка кейсов:   4%|▍         | 5/130 [03:15<1:23:19, 40.00s/it]


💾 Сохранено 5 кейсов


Обработка кейсов:   8%|▊         | 10/130 [06:24<1:16:31, 38.26s/it]


💾 Сохранено 10 кейсов


Обработка кейсов:  12%|█▏        | 15/130 [09:36<1:15:39, 39.47s/it]

...


💾 Сохранено 25 кейсов


Обработка кейсов:  23%|██▎       | 30/130 [18:53<59:54, 35.95s/it]  


💾 Сохранено 30 кейсов


Обработка кейсов:  25%|██▌       | 33/130 [21:08<1:02:07, 38.43s/it]


KeyboardInterrupt: 

In [ ]:
output_file = "test_with_critique.xlsx"
test_df.to_excel(output_file, index=False)